# EXP-029 checkpoint recovery
Select a CPU runtime and Run all. Uses the original Drive checkpoint seals and handoff. Optional L4 mode replays the saved QLoRA checkpoint for the missing shuffle diagnostic. The recovery fits only the original CPU E50+L reference heads to reconstruct baseline probabilities.

In [ ]:
# Leave False for CPU-only recovery. True authorizes only the shuffle replay on L4.
AUTHORIZE_L4_SHUFFLE = False
DRIVE_ROOT = "/content/drive/MyDrive"
SOURCE = "5e5c33cfd8a3d8b2f46b04d7354c758a1c245aef"

from google.colab import drive, runtime
import os
from pathlib import Path
import subprocess
import sys
import tempfile
from uuid import uuid4

drive.mount("/content/drive")
root = Path(DRIVE_ROOT)
persistent = root / "exp029"
destination = persistent / "attempts" / ("task037-recovery-" + uuid4().hex)
repo = "https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git"
env = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", OMP_THREAD_LIMIT="1",
           MKL_NUM_THREADS="1", CUBLAS_WORKSPACE_CONFIG=":4096:8", HF_HUB_DISABLE_XET="1")
env.pop("PYTHONPATH", None)
if not AUTHORIZE_L4_SHUFFLE:
    env["CUDA_VISIBLE_DEVICES"] = "-1"

def run(cmd, timeout=600):
    subprocess.run(cmd, env=env, check=True, timeout=timeout)

try:
    with tempfile.TemporaryDirectory(dir="/content") as tmp:
        path = Path(tmp)
        source = path / "source"
        run(["git", "clone", "--no-checkout", repo, str(source)])
        run(["git", "-C", str(source), "checkout", "--detach", SOURCE])
        run([sys.executable, "-m", "pip", "install", "uv==0.7.13"])
        uv = [sys.executable, "-m", "uv"]
        run([*uv, "python", "install", "3.12.10"])
        run([*uv, "venv", "--python", "3.12.10", str(path / "venv")])
        python = str(path / "venv/bin/python")
        options = ["--python", python]
        if AUTHORIZE_L4_SHUFFLE:
            run([*uv, "pip", "sync", *options, "--extra-index-url", "https://download.pytorch.org/whl/cu126",
                 "--index-strategy", "unsafe-best-match", str(source / "requirements-exp026.lock.txt")])
        else:
            run([*uv, "pip", "sync", *options, str(source / "requirements-exp028-cpu.lock.txt")])
            run([*uv, "pip", "install", *options, "--index-url",
                 "https://download.pytorch.org/whl/cpu", "torch==2.7.1+cpu"])
        cmd = [python, str(source / "scripts/recover_exp029.py"), "--handoff", str(root / "exp023_handoff.zip"),
               "--original-attempt", str(persistent / "attempts/20261119T103915Z-2c837e97"),
               "--models", str(persistent / "models"), "--destination", str(destination)]
        if AUTHORIZE_L4_SHUFFLE:
            cmd += ["--authorized-l4-model-cache", str(persistent / "qwen_assets")]
        run(cmd, timeout=1200)
        print("Verified Drive results:", destination)
finally:
    if AUTHORIZE_L4_SHUFFLE:
        runtime.unassign()
